In [1]:
import os
%pwd

'c:\\Users\\Korisnik\\Desktop\\ml_projects\\summerizer\\TextSummarizer\\research'

In [2]:
os.chdir('../')

In [3]:
%pwd

'c:\\Users\\Korisnik\\Desktop\\ml_projects\\summerizer\\TextSummarizer'

In [4]:
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
from typing import List
from peft import TaskType
from typing import Literal

@dataclass
class TrainingParams:
    num_train_epochs: int
    per_device_train_batch_size: int
    per_device_eval_batch_size: int
    gradient_accumulation_steps: int
    learning_rate: float
    warmup_ratio: float
    weight_decay: float
    max_grad_norm: float

    fp16: bool
    bf16: bool

    logging_steps: int
    evaluation_strategy: Literal["no", "steps", "epoch"]
    eval_steps: int
    save_steps: int
    save_total_limit: int

    predict_with_generate: bool
    generation_max_length: int

    report_to: str
    remove_unused_columns: bool
    label_names: tuple

    ddp_find_unused_parameters: bool
    dataloader_num_workers: int
    local_rank: int = -1

@dataclass
class LoraParameters:
    r: int
    lora_alpha: int
    target_modules: List[str]
    lora_dropout: float
    bias: str
    task_type: TaskType

@dataclass
class ModelTrainerConfig:
    output_dir: Path
    model_ckpt: Path
    data_path: Path
    training_params: TrainingParams
    lora_parameters: LoraParameters

c:\Users\Korisnik\Desktop\ml_projects\summerizer\TextSummarizer\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from src.textsummarizer.constants import *
from src.textsummarizer.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(self, config_path=CONFIG_FILE_PATH, params_file_path=PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_file_path)

        create_directories([self.config.artifact_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        trainingParams = self.params.TrainingArguments
        loraParams = self.params.Lora

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
                output_dir = config.root_dir,
                model_ckpt=config.model_ckpt,
                data_path= config.data_path,

                training_params = TrainingParams(
                    num_train_epochs = trainingParams.num_train_epochs,
                    per_device_train_batch_size = trainingParams.per_device_train_batch_size,
                    per_device_eval_batch_size = trainingParams.per_device_eval_batch_size,
                    gradient_accumulation_steps = trainingParams.gradient_accumulation_steps,
                    learning_rate = trainingParams.learning_rate,
                    warmup_ratio = trainingParams.warmup_ratio,
                    weight_decay = trainingParams.weight_decay,
                    max_grad_norm = trainingParams.max_grad_norm,
                    fp16 = trainingParams.fp16,
                    bf16 = trainingParams.bf16,
                    logging_steps = trainingParams.logging_steps,
                    evaluation_strategy = trainingParams.eval_strategy,
                    eval_steps = trainingParams.eval_steps,
                    save_steps = trainingParams.save_steps,
                    save_total_limit = trainingParams.save_total_limit,
                    predict_with_generate = trainingParams.predict_with_generate,
                    generation_max_length = trainingParams.generation_max_length,
                    report_to = trainingParams.report_to,
                    remove_unused_columns = trainingParams.remove_unused_columns,
                    label_names = trainingParams.label_names,
                    ddp_find_unused_parameters = trainingParams.ddp_find_unused_parameters,
                    dataloader_num_workers = trainingParams.dataloader_num_workers
            ),
            lora_parameters = LoraParameters(
                r = loraParams.r,
                lora_alpha = loraParams.lora_alpha,
                target_modules = loraParams.target_modules,
                lora_dropout = loraParams.lora_dropout,
                bias = loraParams.bias,
                task_type = loraParams.task_type
            )
        )

        return model_trainer_config

In [7]:
import os
import torch
from datasets import load_from_disk
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
class ModelTrainer: 
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
        os.environ["WANDB_DISABLED"] = "true"
        os.environ["TOKENIZERS_PARALLELISM"] = "false"

    def train(self):
        
        local_rank = int(os.environ.get("LOCAL_RANK", -1))
        world_size = int(os.environ.get("WORLD_SIZE", 1))

        is_main_process = local_rank in [-1, 0]

        if is_main_process:
            print(f"GPUs: {torch.cuda.device_count()}, World size: {world_size}, Rank: {local_rank}")

        model_name = self.config.model_ckpt
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        # FIX: fp32 za stabilnost — Pegasus ima overflow probleme u fp16
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            torch_dtype=torch.float32,
        )
        model.config.use_cache = False

        lora_config = LoraConfig(
            r = self.config.lora_parameters.r,
            lora_alpha = self.config.lora_parameters.lora_alpha,
            target_modules = self.config.lora_parameters.target_modules,
            lora_dropout = self.config.lora_parameters.lora_dropout,
            bias= self.config.lora_parameters.bias,
            task_type = self.config.lora_parameters.task_type,
        )
        model = get_peft_model(model, lora_config)

        if is_main_process:
            model.print_trainable_parameters()

        dataset_samsum = load_from_disk(self.config.data_path)

        data_collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer, model=model,
            padding=True, label_pad_token_id=-100,
        )

        training_args = Seq2SeqTrainingArguments(
            output_dir="/kaggle/working/pegasus-lora",
            num_train_epochs=1,
            per_device_train_batch_size=2,      # manji batch za stabilnost
            per_device_eval_batch_size=2,
            gradient_accumulation_steps=8,
            learning_rate=3e-5,                 # FIX: manji LR (1e-4 → 3e-5)
            warmup_ratio=0.05,                  # FIX: ratio umjesto steps, sigurniji
            weight_decay=0.01,
            max_grad_norm=1.0,
            fp16=False,                         # FIX: isključen fp16
            bf16=False,
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=200,
            save_steps=500,
            save_total_limit=2,
            predict_with_generate=True,
            generation_max_length=128,
            report_to="none",
            remove_unused_columns=False,
            label_names=["labels"],
            ddp_find_unused_parameters=False,
            dataloader_num_workers=2,
        )

        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer = Seq2SeqTrainer(
            model=model,
            args=training_args,
            processing_class=tokenizer,
            data_collator=data_collator,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"],
        )

        trainer.train()

        trainer.save_model("/kaggle/working/pegasus-lora-adapter")
        tokenizer.save_pretrained("/kaggle/working/pegasus-lora-adapter")
        print("Saved!")

        merged_model = model.merge_and_unload()
        print(type(merged_model))  # sad je čisti PegasusForConditionalGeneration

        # Snimi merged model
        merged_model.save_pretrained("/kaggle/working/pegasus-lora-merged")
        tokenizer.save_pretrained("/kaggle/working/pegasus-lora-merged")

In [ ]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e

[2026-05-20 16:19:09,799: INFO: common]: YAML file config\config.yaml loaded successfully.
[2026-05-20 16:19:09,803: INFO: common]: YAML file params.yaml loaded successfully.
[2026-05-20 16:19:09,804: INFO: common]: Directory created at: artifacts
[2026-05-20 16:19:09,808: INFO: common]: Directory created at: artifacts/model_trainer
GPUs: 0, World size: 1, Rank: -1
[2026-05-20 16:19:10,093: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-05-20 16:19:10,161: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"
[2026-05-20 16:19:10,331: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"


[2026-05-20 16:19:10,337: WARNING: _http]: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[2026-05-20 16:19:10,381: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"
[2026-05-20 16:19:10,541: INFO: _client]: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
[2026-05-20 16:19:10,693: INFO: _client]: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
[2026-05-20 16:19:11,258: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-05-20 16:19:11,281: INFO: _cli

`torch_dtype` is deprecated! Use `dtype` instead!


[2026-05-20 16:19:11,716: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-05-20 16:19:11,741: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"
[2026-05-20 16:19:11,890: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
[2026-05-20 16:19:12,039: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"
[2026-05-20 16:19:12,193: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"
[2026-05-20 16:19:12,395: INFO: _client]: HTTP Request: GET https://huggingface.co/api/models/google/pega